# Tutorial 1: Understanding the PSD → GP → SFH Model

This notebook walks through the core idea behind `diffsed`: modeling galaxy star formation histories as **continuous correlated fields** governed by a power spectral density (PSD).

## What you'll learn

1. **Power Spectral Density (PSD)**: What it means physically for star formation burstiness
2. **Gaussian Process (GP)**: How we generate correlated SFH fluctuations from the PSD
3. **Mean SFH**: The smooth secular envelope (double power law)
4. **Full SFH**: Combining mean + GP fluctuations with the lognormal correction
5. **How parameters affect the SFH**: Interactive exploration

## The key idea

Traditional SED fitting uses either parametric SFHs (too rigid) or binned non-parametric SFHs (arbitrary bin choices). We instead model the SFH as:

$$\text{SFR}(t) = \overline{\text{SFR}}(t) \cdot \exp\bigl(x(t) - \sigma_x^2/2\bigr)$$

where $\overline{\text{SFR}}(t)$ is a smooth mean (double power law) and $x(t)$ is a Gaussian Process whose temporal correlations are set by the PSD. The PSD encodes the **amplitude and timescale of burstiness** — different feedback mechanisms produce different PSDs.

In [1]:
# Setup: imports and JAX configuration
import sys
sys.path.insert(0, "../src")

from diffsed.utils.devices import setup_jax, print_device_info
setup_jax()
print_device_info()

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

# Use a clean style
plt.rcParams.update({
    "figure.figsize": (10, 4),
    "font.size": 12,
    "axes.linewidth": 1.2,
    "xtick.direction": "in",
    "ytick.direction": "in",
})

JAX platform:    CPU
Devices:         1x TFRT_CPU_0
64-bit:          Yes
JAX version:     0.9.1


## Part 1: The Power Spectral Density (PSD)

The PSD tells you **how much variability** exists at each timescale. We use the **damped random walk** (DRW), which has two parameters:

| Parameter | Symbol | Meaning |
|-----------|--------|---------|
| Amplitude | $\sigma_\text{PS}$ | How bursty (larger = wilder fluctuations) |
| Timescale | $\tau_\text{PS}$ | Coherence time of bursts (longer = smoother variations) |

$$P(\omega) = \frac{\sigma_\text{PS}^2 \, \tau_\text{PS}}{1 + (\tau_\text{PS} \, \omega)^2}$$

At low frequencies ($\omega \ll 1/\tau$), the PSD is flat: $P \approx \sigma^2 \tau$ (white noise).  
At high frequencies ($\omega \gg 1/\tau$), the PSD falls as $1/\omega^2$ (correlated, smooth).  
The **knee** at $\omega = 1/\tau$ separates these regimes.

In [2]:
from diffsed.models.sfh.psd_models import psd_drw, drw_variance, drw_acf

# Define 4 physically motivated regimes (from our paper Table 1)
regimes = {
    "Smooth":        {"sigma_ps": 0.5, "tau_ps": 200e6, "color": "#2166ac"},
    "Moderate":      {"sigma_ps": 1.0, "tau_ps": 50e6,  "color": "#67a9cf"},
    "Bursty":        {"sigma_ps": 2.0, "tau_ps": 20e6,  "color": "#ef8a62"},
    "Highly bursty": {"sigma_ps": 3.0, "tau_ps": 5e6,   "color": "#b2182b"},
}

# Frequency grid (in Myr^-1 for readability)
omega = jnp.logspace(-4, 1, 500)  # rad/Myr

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: PSD P(omega)
ax = axes[0]
for name, r in regimes.items():
    # Convert tau from yr to Myr for this plot
    tau_myr = r["tau_ps"] / 1e6
    p = psd_drw(omega, r["sigma_ps"], tau_myr)
    ax.loglog(omega, p, label=name, color=r["color"], lw=2.5)
    # Mark the knee
    ax.axvline(1.0 / tau_myr, color=r["color"], ls=":", alpha=0.4)

ax.set_xlabel(r"Angular frequency $\omega$ [rad Myr$^{-1}$]")
ax.set_ylabel(r"$P(\omega)$")
ax.set_title("Power Spectral Density (DRW)")
ax.legend(fontsize=10)
ax.set_xlim(1e-4, 10)

# Right: Autocorrelation function xi(dt)
ax = axes[1]
dt_myr = jnp.linspace(0, 500, 300)  # Myr

for name, r in regimes.items():
    tau_myr = r["tau_ps"] / 1e6
    acf = drw_acf(dt_myr, r["sigma_ps"], tau_myr)
    # Normalize to ACF(0) for shape comparison
    acf_norm = acf / acf[0]
    ax.plot(dt_myr, acf_norm, label=name, color=r["color"], lw=2.5)

ax.set_xlabel(r"Time lag $\Delta t$ [Myr]")
ax.set_ylabel(r"Normalized ACF $\xi(\Delta t) / \xi(0)$")
ax.set_title("Autocorrelation Function")
ax.legend(fontsize=10)
ax.set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.show()

# Print variance for each regime
print("Process variance sigma_x^2 = sigma_PS^2 / 2:")
for name, r in regimes.items():
    var = drw_variance(r["sigma_ps"])
    print(f"  {name:16s}: sigma_x^2 = {var:.3f}  (sigma_x = {var**0.5:.3f} dex in log SFR)")

W0313 14:56:25.593653 2011648 cpp_gen_intrinsics.cc:74] Empty bitcode string provided for eigen. Optimizations relying on this IR will be disabled.


Process variance sigma_x^2 = sigma_PS^2 / 2:
  Smooth          : sigma_x^2 = 0.125  (sigma_x = 0.354 dex in log SFR)
  Moderate        : sigma_x^2 = 0.500  (sigma_x = 0.707 dex in log SFR)
  Bursty          : sigma_x^2 = 2.000  (sigma_x = 1.414 dex in log SFR)
  Highly bursty   : sigma_x^2 = 4.500  (sigma_x = 2.121 dex in log SFR)


/var/folders/km/_d_w3tds0hs4c5pbvflcdt480000gn/T/ipykernel_39975/4169615543.py:50: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### What we just saw

- **Left panel**: The PSD shows how much power (variability) exists at each frequency. The "knee" at $\omega = 1/\tau_\text{PS}$ separates correlated (low-$\omega$) from uncorrelated (high-$\omega$) variations. Bursty galaxies have more power at all frequencies.

- **Right panel**: The ACF shows how correlated the SFR is between two times separated by $\Delta t$. Smooth galaxies stay correlated for hundreds of Myr; highly bursty ones decorrelate within ~5 Myr.

- **Physical interpretation**: $\sigma_\text{PS}$ controls the **amplitude** of SFR fluctuations (in dex of log SFR), while $\tau_\text{PS}$ controls the **memory** — how long a burst or lull persists. Different feedback mechanisms produce different ($\sigma$, $\tau$) combinations: stellar winds (~1-10 Myr), SN feedback (~20-50 Myr), gas accretion (~100+ Myr).

---

## Part 2: Generating GP Realizations from the PSD

The IFT (Information Field Theory) trick: generate a correlated field $x(t)$ from the PSD using the **correlated field model**:

$$x = \text{IFFT}\!\left(\sqrt{P} \cdot \boldsymbol{\xi}\right), \qquad \boldsymbol{\xi} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$$

The latent vector $\boldsymbol{\xi}$ is **standardized** — it's just i.i.d. standard normal noise. All the temporal correlations come from multiplying by $\sqrt{P(\omega)}$ in Fourier space. This is the key to efficient inference: samplers explore the simple $\xi$-space while the physics is encoded in the amplitude operator $\sqrt{P}$.

In [3]:
from diffsed.models.sfh.gp_sfh import (
    gp_from_xi, generate_gp_batch, compute_sqrt_power_drw
)
from diffsed.utils.grid import make_log_age_grid, grid_spacing, log_age_to_age_yr

# Our time grid: 256 points uniform in log10(age/yr), from 1 Myr to 13.8 Gyr
N_GRID = 256
log_age_grid = make_log_age_grid(N_GRID)
d_log_age = grid_spacing(log_age_grid)
age_yr = log_age_to_age_yr(log_age_grid)
age_gyr = age_yr / 1e9

print(f"Grid: {N_GRID} points from {10**log_age_grid[0]/1e6:.1f} Myr to {10**log_age_grid[-1]/1e9:.1f} Gyr")
print(f"Spacing: {float(d_log_age):.4f} dex")
print(f"Resolution at 10 Myr: ~{10**(7+float(d_log_age))/1e6 - 10:.1f} Myr between points")
print(f"Resolution at 1 Gyr:  ~{10**(9+float(d_log_age))/1e9 - 1:.2f} Gyr between points")
print()
print("This is the beauty of log-age grids: finer resolution where the SED is most sensitive (young ages).")

Grid: 256 points from 1.0 Myr to 13.8 Gyr
Spacing: 0.0162 dex
Resolution at 10 Myr: ~0.4 Myr between points
Resolution at 1 Gyr:  ~0.04 Gyr between points

This is the beauty of log-age grids: finer resolution where the SED is most sensitive (young ages).


In [4]:
# Generate GP realizations for all 4 regimes
# Step 1: compute the amplitude operator sqrt(P/d) for each regime
# Step 2: draw random xi ~ N(0, I) 
# Step 3: apply x = IFFT(sqrt(P) * xi)

key = jax.random.PRNGKey(42)
n_realizations = 5  # draw 5 random SFHs per regime

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)

for ax, (name, r) in zip(axes.flat, regimes.items()):
    # Step 1: amplitude operator (with Jacobian correction for log-age grid)
    sqrt_power = compute_sqrt_power_drw(
        N_GRID, float(d_log_age), r["sigma_ps"], r["tau_ps"]
    )
    
    # Step 2 & 3: draw GP realizations
    batch = generate_gp_batch(key, sqrt_power, N_GRID, n_realizations)
    
    # Plot each realization
    for i in range(n_realizations):
        ax.plot(age_gyr, batch[i], alpha=0.6, lw=1.2, color=r["color"])
    
    # Mark zero line
    ax.axhline(0, color="k", ls="--", alpha=0.3, lw=0.8)
    
    ax.set_title(f"{name}: $\\sigma_\\mathrm{{PS}}$={r['sigma_ps']}, "
                 f"$\\tau_\\mathrm{{PS}}$={r['tau_ps']/1e6:.0f} Myr",
                 fontsize=11)
    ax.set_ylabel("$x(t)$ [log SFR fluctuation]")
    ax.set_xscale("log")
    
    # Set y-limits based on expected variance
    var = drw_variance(r["sigma_ps"])
    ylim = max(3 * var**0.5, 0.5)
    ax.set_ylim(-ylim, ylim)

axes[1, 0].set_xlabel("Lookback time [Gyr]")
axes[1, 1].set_xlabel("Lookback time [Gyr]")

plt.suptitle("GP realizations $x(t)$ from DRW PSD — the stochastic part of the SFH",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print("Key observation: same xi ~ N(0,I) but different sqrt(P) produces")
print("very different correlation structures. The PSD IS the prior.")

Key observation: same xi ~ N(0,I) but different sqrt(P) produces
very different correlation structures. The PSD IS the prior.


/var/folders/km/_d_w3tds0hs4c5pbvflcdt480000gn/T/ipykernel_39975/1891143092.py:44: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Under the hood: what `gp_from_xi` does

Let's trace through the computation step by step:

```python
# 1. Your latent vector (what the sampler explores)
xi = random.normal(key, shape=(256,))           # 256 i.i.d. N(0,1) values

# 2. Convert to Hermitian-symmetric Fourier coefficients
xi_hat = xi_to_complex(xi, 256)                 # 129 complex values

# 3. Multiply by amplitude operator (encodes the PSD)
coeffs = sqrt_power * xi_hat                     # colored noise in Fourier space

# 4. Inverse FFT back to time domain
x = jnp.fft.irfft(coeffs, n=256)               # correlated GP realization
```

This is identical to NIFTy.re's correlated field model: $s = \mathcal{F}^{-1}(\sqrt{P} \cdot \xi)$.

---

## Part 3: The Mean Star Formation History

The GP $x(t)$ fluctuates around zero — it only generates the **stochastic** part. The overall shape of the SFH (rising at early times, declining at late times) comes from a separate **mean SFH** model.

We use the BAGPIPES-style **double power law**:

$$\overline{\text{SFR}}(t) = \frac{A}{\left(\frac{t}{\tau}\right)^\alpha + \left(\frac{t}{\tau}\right)^{-\beta}}$$

| Parameter | Meaning |
|-----------|---------|
| $\alpha$ | Falling slope at late times ($t \gg \tau$) |
| $\beta$ | Rising slope at early times ($t \ll \tau$) |
| $\tau$ | Turnover time (roughly when SFR peaks) |
| $A$ | Overall normalization |

In [5]:
from diffsed.models.sfh.mean_sfh import double_powerlaw

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: varying alpha (falling slope)
ax = axes[0]
for alpha, ls in zip([0.5, 1.0, 2.0, 4.0], ["-", "--", "-.", ":"]):
    sfr = double_powerlaw(age_yr, alpha=alpha, beta=1.0, tau=2e9, norm=10.0)
    ax.plot(age_gyr, sfr, label=fr"$\alpha$={alpha}", lw=2, ls=ls)
ax.set_xlabel("Lookback time [Gyr]")
ax.set_ylabel("SFR [M$_\odot$/yr]")
ax.set_title(r"Varying $\alpha$ (falling slope), $\beta$=1, $\tau$=2 Gyr")
ax.set_xscale("log")
ax.legend()
ax.set_xlim(0.001, 14)

# Right: varying tau (turnover time)
ax = axes[1]
for tau_gyr, color in zip([0.5, 1.0, 3.0, 8.0], 
                           ["#d73027", "#fc8d59", "#91bfdb", "#4575b4"]):
    sfr = double_powerlaw(age_yr, alpha=1.5, beta=0.8, tau=tau_gyr*1e9, norm=10.0)
    ax.plot(age_gyr, sfr, label=fr"$\tau$={tau_gyr} Gyr", lw=2, color=color)
ax.set_xlabel("Lookback time [Gyr]")
ax.set_ylabel("SFR [M$_\odot$/yr]")
ax.set_title(r"Varying $\tau$ (peak time), $\alpha$=1.5, $\beta$=0.8")
ax.set_xscale("log")
ax.legend()
ax.set_xlim(0.001, 14)

plt.tight_layout()
plt.show()

print("The double power law provides the secular 'envelope' of the SFH.")
print("The GP fluctuations will be added ON TOP of this smooth component.")

<>:11: SyntaxWarning: invalid escape sequence '\o'
<>:24: SyntaxWarning: invalid escape sequence '\o'
<>:11: SyntaxWarning: invalid escape sequence '\o'
<>:24: SyntaxWarning: invalid escape sequence '\o'
/var/folders/km/_d_w3tds0hs4c5pbvflcdt480000gn/T/ipykernel_39975/2098608907.py:11: SyntaxWarning: invalid escape sequence '\o'
  ax.set_ylabel("SFR [M$_\odot$/yr]")
/var/folders/km/_d_w3tds0hs4c5pbvflcdt480000gn/T/ipykernel_39975/2098608907.py:24: SyntaxWarning: invalid escape sequence '\o'
  ax.set_ylabel("SFR [M$_\odot$/yr]")


The double power law provides the secular 'envelope' of the SFH.
The GP fluctuations will be added ON TOP of this smooth component.


/var/folders/km/_d_w3tds0hs4c5pbvflcdt480000gn/T/ipykernel_39975/2098608907.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Part 4: The Full SFH — Mean + GP with Lognormal Correction

Now we combine everything. The full SFH is:

$$\text{SFR}(t) = \overline{\text{SFR}}(t) \cdot \exp\!\left(x(t) - \frac{\sigma_x^2}{2}\right)$$

**Why the $-\sigma_x^2/2$ correction?** The GP fluctuations $x(t)$ live in log space. If we just did $\overline{\text{SFR}} \cdot e^{x(t)}$, the *expected* SFR would be:

$$\langle\text{SFR}\rangle = \overline{\text{SFR}} \cdot \langle e^x \rangle = \overline{\text{SFR}} \cdot e^{\sigma_x^2/2}$$

which is **biased high** (because the exponential of a zero-mean Gaussian has mean $> 1$). The correction term $-\sigma_x^2/2$ ensures that $\langle\text{SFR}\rangle = \overline{\text{SFR}}$ — the mean SFH retains its intended interpretation as the *average* star formation rate.

In [6]:
# Full SFH: mean * exp(GP - correction)
# Let's show the same galaxy under different burstiness levels

# Fixed mean SFH parameters (a typical star-forming galaxy)
sfr_mean = double_powerlaw(age_yr, alpha=1.5, beta=0.8, tau=2e9, norm=5.0)

# Fixed random seed -> same "underlying galaxy", different burstiness
key = jax.random.PRNGKey(123)
xi = jax.random.normal(key, shape=(N_GRID,))

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True, sharey=True)

for ax, (name, r) in zip(axes.flat, regimes.items()):
    sqrt_power = compute_sqrt_power_drw(
        N_GRID, float(d_log_age), r["sigma_ps"], r["tau_ps"]
    )
    
    # Generate GP from the SAME xi (different amplitude operator)
    gp_x = gp_from_xi(xi, sqrt_power, N_GRID)
    
    # Lognormal correction
    k0_half = drw_variance(r["sigma_ps"]) / 2.0
    
    # Full SFH
    sfr_full = sfr_mean * jnp.exp(gp_x - k0_half)
    
    # Plot
    ax.fill_between(age_gyr, 0, sfr_mean, alpha=0.15, color="gray", label="Mean SFH")
    ax.plot(age_gyr, sfr_mean, color="gray", ls="--", lw=1.5, alpha=0.7)
    ax.plot(age_gyr, sfr_full, color=r["color"], lw=1.5, label="Full SFH")
    
    ax.set_title(f"{name}: $\\sigma_\\mathrm{{PS}}$={r['sigma_ps']}, "
                 f"$\\tau_\\mathrm{{PS}}$={r['tau_ps']/1e6:.0f} Myr")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_ylim(0.01, 500)
    ax.legend(fontsize=9, loc="upper left")

axes[1, 0].set_xlabel("Lookback time [Gyr]")
axes[1, 1].set_xlabel("Lookback time [Gyr]")
axes[0, 0].set_ylabel("SFR [M$_\odot$/yr]")
axes[1, 0].set_ylabel("SFR [M$_\odot$/yr]")

plt.suptitle("Full SFH = Mean (gray) + GP fluctuations (colored)", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

<>:41: SyntaxWarning: invalid escape sequence '\o'
<>:42: SyntaxWarning: invalid escape sequence '\o'
<>:41: SyntaxWarning: invalid escape sequence '\o'
<>:42: SyntaxWarning: invalid escape sequence '\o'
/var/folders/km/_d_w3tds0hs4c5pbvflcdt480000gn/T/ipykernel_39975/3448623847.py:41: SyntaxWarning: invalid escape sequence '\o'
  axes[0, 0].set_ylabel("SFR [M$_\odot$/yr]")
/var/folders/km/_d_w3tds0hs4c5pbvflcdt480000gn/T/ipykernel_39975/3448623847.py:42: SyntaxWarning: invalid escape sequence '\o'
  axes[1, 0].set_ylabel("SFR [M$_\odot$/yr]")


/var/folders/km/_d_w3tds0hs4c5pbvflcdt480000gn/T/ipykernel_39975/3448623847.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Part 5: Gradients — The Whole Point of Differentiability

Everything we've built is differentiable. Let's verify that gradients flow through the entire PSD → GP → SFH pipeline and that they're **physically meaningful**.

In [7]:
# Demonstrate end-to-end gradients through the SFH pipeline

def sfh_pipeline(xi, sigma_ps, tau_ps, alpha, beta, tau_sfh, norm):
    """Full SFH from latent variables to SFR(t)."""
    sqrt_power = compute_sqrt_power_drw(N_GRID, float(d_log_age), sigma_ps, tau_ps)
    gp_x = gp_from_xi(xi, sqrt_power, N_GRID)
    k0_half = drw_variance(sigma_ps) / 2.0
    sfr_mean = double_powerlaw(age_yr, alpha, beta, tau_sfh, norm)
    sfr = sfr_mean * jnp.exp(gp_x - k0_half)
    return sfr

# Compute total stellar mass formed (integral of SFR * dt) as our scalar objective
def total_mass(xi, sigma_ps, tau_ps, alpha, beta, tau_sfh, norm):
    sfr = sfh_pipeline(xi, sigma_ps, tau_ps, alpha, beta, tau_sfh, norm)
    dt = jnp.diff(age_yr, prepend=0.0)
    return jnp.sum(sfr * dt)

# Compute gradients w.r.t. ALL parameters simultaneously
xi = jax.random.normal(jax.random.PRNGKey(0), shape=(N_GRID,))

grad_fn = jax.grad(total_mass, argnums=(0, 1, 2, 3, 4, 5, 6))
grads = grad_fn(xi, 1.0, 50e6, 1.5, 0.8, 2e9, 5.0)

param_names = ["xi (256-dim)", "sigma_PS", "tau_PS", "alpha", "beta", "tau_SFH", "norm"]
print("Gradients of total stellar mass w.r.t. each parameter:")
print("=" * 60)
for name, g in zip(param_names, grads):
    if isinstance(g, jnp.ndarray) and g.ndim > 0:
        print(f"  d(M*)/d({name:12s}): shape={g.shape}, "
              f"RMS={float(jnp.sqrt(jnp.mean(g**2))):.4e}")
    else:
        print(f"  d(M*)/d({name:12s}): {float(g):+.4e}")

print()
print("All gradients are finite and non-zero!")
print("This means we can use gradient-based samplers (HMC, NUTS, geoVI)")
print("to efficiently explore this parameter space.")

Gradients of total stellar mass w.r.t. each parameter:
  d(M*)/d(xi (256-dim)): shape=(256,), RMS=1.9446e+07
  d(M*)/d(sigma_PS    ): -5.5881e+09
  d(M*)/d(tau_PS      ): -3.2867e+00
  d(M*)/d(alpha       ): -6.1300e+09
  d(M*)/d(beta        ): -1.1116e+09
  d(M*)/d(tau_SFH     ): +3.9006e+00
  d(M*)/d(norm        ): +2.0730e+09

All gradients are finite and non-zero!
This means we can use gradient-based samplers (HMC, NUTS, geoVI)
to efficiently explore this parameter space.


## Part 6: Sensitivity — Which Parameters Matter Most?

Let's visualize how the SFH responds to small perturbations in each parameter. This is effectively what the gradient tells us, but shown as actual SFH curves.

In [8]:
# Show sensitivity: perturb one parameter at a time

xi_fixed = jax.random.normal(jax.random.PRNGKey(77), shape=(N_GRID,))
base_params = dict(sigma_ps=1.0, tau_ps=50e6, alpha=1.5, beta=0.8, tau_sfh=2e9, norm=5.0)

def compute_sfr(**params):
    sqrt_power = compute_sqrt_power_drw(N_GRID, float(d_log_age), 
                                         params["sigma_ps"], params["tau_ps"])
    gp_x = gp_from_xi(xi_fixed, sqrt_power, N_GRID)
    k0_half = drw_variance(params["sigma_ps"]) / 2.0
    sfr_mean = double_powerlaw(age_yr, params["alpha"], params["beta"], 
                                params["tau_sfh"], params["norm"])
    return sfr_mean * jnp.exp(gp_x - k0_half)

sfr_base = compute_sfr(**base_params)

# Parameters to vary, with their perturbation ranges
variations = [
    ("sigma_ps", [0.3, 0.5, 1.0, 2.0, 3.0], r"$\sigma_\mathrm{PS}$ (burstiness)"),
    ("tau_ps",   [5e6, 20e6, 50e6, 100e6, 200e6], r"$\tau_\mathrm{PS}$ (coherence time)"),
    ("alpha",    [0.5, 1.0, 1.5, 2.5, 4.0], r"$\alpha$ (late-time decline)"),
    ("norm",     [1.0, 3.0, 5.0, 10.0, 20.0], r"$A$ (SFR normalization)"),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
cmap = plt.cm.viridis

for ax, (param, values, label) in zip(axes.flat, variations):
    for i, val in enumerate(values):
        params = {**base_params, param: val}
        sfr = compute_sfr(**params)
        color = cmap(i / (len(values) - 1))
        
        if isinstance(val, float) and val > 1e4:
            val_str = f"{val/1e6:.0f} Myr"
        else:
            val_str = f"{val}"
        ax.plot(age_gyr, sfr, color=color, lw=1.8, label=f"{val_str}")
    
    ax.set_title(label, fontsize=12)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_ylim(0.01, 500)
    ax.legend(fontsize=8, ncol=2)

axes[1, 0].set_xlabel("Lookback time [Gyr]")
axes[1, 1].set_xlabel("Lookback time [Gyr]")
axes[0, 0].set_ylabel("SFR [M$_\odot$/yr]")
axes[1, 0].set_ylabel("SFR [M$_\odot$/yr]")

plt.suptitle("Parameter sensitivity: how each parameter affects the SFH", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

<>:48: SyntaxWarning: invalid escape sequence '\o'
<>:49: SyntaxWarning: invalid escape sequence '\o'
<>:48: SyntaxWarning: invalid escape sequence '\o'
<>:49: SyntaxWarning: invalid escape sequence '\o'
/var/folders/km/_d_w3tds0hs4c5pbvflcdt480000gn/T/ipykernel_39975/2442762705.py:48: SyntaxWarning: invalid escape sequence '\o'
  axes[0, 0].set_ylabel("SFR [M$_\odot$/yr]")
/var/folders/km/_d_w3tds0hs4c5pbvflcdt480000gn/T/ipykernel_39975/2442762705.py:49: SyntaxWarning: invalid escape sequence '\o'
  axes[1, 0].set_ylabel("SFR [M$_\odot$/yr]")


/var/folders/km/_d_w3tds0hs4c5pbvflcdt480000gn/T/ipykernel_39975/2442762705.py:53: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Summary

In this tutorial you learned the core components of the `diffsed` SFH model:

| Component | What it does | Parameters |
|-----------|-------------|------------|
| **PSD** (DRW) | Defines the frequency structure of burstiness | $\sigma_\text{PS}$ (amplitude), $\tau_\text{PS}$ (timescale) |
| **GP** | Generates correlated fluctuations via $x = \text{IFFT}(\sqrt{P} \cdot \xi)$ | $\xi \sim \mathcal{N}(0, I)$ — the latent variables samplers explore |
| **Mean SFH** | Smooth secular envelope (double power law) | $\alpha, \beta, \tau, A$ |
| **Lognormal correction** | $-\sigma_x^2/2$ keeps $\langle\text{SFR}\rangle = \overline{\text{SFR}}$ | Automatic from $\sigma_\text{PS}$ |

**Key concepts:**
1. The PSD is the **prior** on temporal correlations — it encodes feedback physics
2. The latent vector $\xi$ is **standardized** ($\sim \mathcal{N}(0, I)$), making inference efficient
3. Everything is **differentiable** — gradients flow from any observable back to all parameters
4. The model naturally separates **secular evolution** (mean SFH) from **stochastic burstiness** (GP)

**Next:** [Tutorial 2](02_forward_model.ipynb) shows how SFH → SED via dust and DSPS, and how to compute photometry.

---

## Appendix: Hardware Resources and Performance

`diffsed` is designed to use whatever hardware is available — from a MacBook to a multi-GPU cluster. Here's how it works and what to expect.

### Will it use my hardware properly?

| Platform | What happens | Setup |
|----------|-------------|-------|
| **Mac (CPU)** | Uses all CPU cores (Apple Silicon or Intel). JAX allocates memory on demand. | `setup_jax()` — nothing special needed |
| **GPU workstation** | Uses CUDA GPU. Memory allocated on demand (no hogging). | `setup_jax()` auto-detects. Install `jax[cuda12]` |
| **HPC cluster** | Multiple GPUs, pre-allocation for speed | `setup_jax(preallocate_gpu=True, gpu_memory_fraction=0.9)` |
| **Cloud (Colab/AWS)** | GPU if available, CPU fallback | `setup_jax()` auto-detects |

### Key environment variables

| Variable | What it does | Default |
|----------|-------------|---------|
| `JAX_ENABLE_X64` | 64-bit precision (important for SED fitting) | `True` via `setup_jax()` |
| `XLA_PYTHON_CLIENT_PREALLOCATE` | Pre-allocate all GPU memory at startup | `false` (allocate on demand) |
| `XLA_PYTHON_CLIENT_MEM_FRACTION` | Fraction of GPU memory to use | JAX default (90%) |
| `JAX_PLATFORMS` | Force platform: `cpu`, `gpu`, `tpu` | Auto-detect |

### Common gotcha: GPU available but not used

If you have a GPU but JAX is using CPU, you'll see a warning from `check_resources()`. The fix:
```bash
pip install jax[cuda12]   # for CUDA 12
# or
pip install jax[cuda11]   # for CUDA 11
```

In [9]:
# Run the resource diagnostic
from diffsed.utils.devices import check_resources, get_n_parallel_chains

check_resources()

print()
print("For fitting galaxies in parallel (e.g., with BlackJAX HMC),")
print(f"this machine can run ~{get_n_parallel_chains()} independent chains simultaneously.")
print()
print("On a GPU with 40GB memory, you could run ~600 chains in parallel,")
print("fitting ~1000 galaxies per GPU-minute (Zacharegkas+2025).")

JAX platform:    CPU
Devices:         1x TFRT_CPU_0
64-bit:          Yes
JAX version:     0.9.1
Compute test:    OK


---
Recommended parallel chains: 14

For fitting galaxies in parallel (e.g., with BlackJAX HMC),
this machine can run ~14 independent chains simultaneously.

On a GPU with 40GB memory, you could run ~600 chains in parallel,
fitting ~1000 galaxies per GPU-minute (Zacharegkas+2025).
